<a href="https://colab.research.google.com/github/braysonimbambi455/CSC4110-G03/blob/main/G03_U9_1_packaging_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSC 4110 — Task 9.1: Model Packaging and Web Application Deployment

**Owner:** Austin Gatita  
**Group:** G03 — Crop Yield  
**Module:** 9 — Model Packaging and Deployment

This notebook uses the group's Task 2.2 preprocessing workflow and the `clean_crops.csv` dataset. It trains the selected Ridge regression model as a complete pipeline, saves and reloads the model with Joblib, and creates a Streamlit application for crop-yield prediction.

The model uses the same engineered features as Task 2.2: rainfall, pesticide use, average temperature, area frequency, crop one-hot encoded columns, and years since 1990. The chronological split uses years through 2009 for training and 2010–2013 for testing.

**Required evidence:** `final_pipe`, `models/G03_model.joblib`, `app/app.py`, `app/requirements.txt`, and `outputs/app_screenshot.png`.

## 1. Mount Google Drive and define project paths

In [45]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import json
import numpy as np
import pandas as pd

# Set the project folder used by the group.
BASE = Path('/content/drive/MyDrive/CSC4110_G03 (1)')
DATA_DIR = BASE / 'data'
OUTPUT_DIR = BASE / 'outputs'
MODEL_DIR = BASE / 'models'
APP_DIR = BASE / 'app'

for folder in [OUTPUT_DIR, MODEL_DIR, APP_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

STAGE1_PATH = DATA_DIR / 'stage1_crops.csv'
CLEAN_PATH = DATA_DIR / 'clean_crops.csv'

print('Project folder:', BASE)
print('Stage 1 dataset exists:', STAGE1_PATH.exists())
print('Clean dataset exists:', CLEAN_PATH.exists())

if not CLEAN_PATH.exists():
    raise FileNotFoundError(f'Could not find {CLEAN_PATH}')

if not STAGE1_PATH.exists():
    raise FileNotFoundError(
        f'Could not find {STAGE1_PATH}. Task 2.2 uses this file to recreate '
        'the country-to-Area_frequency mapping for the Streamlit application.'
    )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project folder: /content/drive/MyDrive/CSC4110_G03 (1)
Stage 1 dataset exists: True
Clean dataset exists: True


## 2. Load the datasets and inspect the columns

In [46]:
# Load the cleaned dataset and the original Stage 1 dataset.
df = pd.read_csv(CLEAN_PATH)
df_stage1 = pd.read_csv(STAGE1_PATH)

print('Clean dataset shape:', df.shape)
print('Clean dataset columns:', df.columns.tolist())
print('Stage 1 dataset shape:', df_stage1.shape)

required_clean_columns = [
    'Year', 'yield_hg_ha', 'log_yield',
    'rain_mm', 'pesticides_t', 'avg_temp', 'Area_frequency'
]
missing_columns = [col for col in required_clean_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f'Missing columns in clean_crops.csv: {missing_columns}')

if not {'Area', 'Item'}.issubset(df_stage1.columns):
    raise ValueError('stage1_crops.csv must contain Area and Item columns.')

display(df.head())

Clean dataset shape: (25932, 19)
Clean dataset columns: ['Item', 'Year', 'yield_hg_ha', 'rain_mm', 'pesticides_t', 'avg_temp', 'log_yield', 'Area_frequency', 'Item_Cassava', 'Item_Maize', 'Item_Plantains and others', 'Item_Potatoes', 'Item_Rice, paddy', 'Item_Sorghum', 'Item_Soybeans', 'Item_Sweet potatoes', 'Item_Wheat', 'Item_Yams', 'years_since_1990']
Stage 1 dataset shape: (25932, 7)


,Item,Year,yield_hg_ha,rain_mm,pesticides_t,avg_temp,log_yield,Area_frequency,Item_Cassava,Item_Maize,Item_Plantains and others,Item_Potatoes,"Item_Rice, paddy",Item_Sorghum,Item_Soybeans,Item_Sweet potatoes,Item_Wheat,Item_Yams,years_since_1990
0,Maize,1990,36613,1485.0,121.0,16.37,10.508159,0.003818,0,1,0,0,0,0,0,0,0,0,0
1,Potatoes,1990,66667,1485.0,121.0,16.37,11.107465,0.003818,0,0,0,1,0,0,0,0,0,0,0
2,"Rice, paddy",1990,23333,1485.0,121.0,16.37,10.057624,0.003818,0,0,0,0,1,0,0,0,0,0,0
3,Sorghum,1990,12500,1485.0,121.0,16.37,9.433484,0.003818,0,0,0,0,0,1,0,0,0,0,0
4,Soybeans,1990,7000,1485.0,121.0,16.37,8.853665,0.003818,0,0,0,0,0,0,1,0,0,0,0


## 3. Recreate the feature columns used in Task 2.2

In [47]:
# Task 2.2 removed Item, Area, Year, and the original yield from the model inputs.
# Crop categories are represented by the existing Item one-hot encoded columns.
item_dummy_cols = [col for col in df.columns if col.startswith('Item_')]

excluded_columns = ['log_yield', 'yield_hg_ha', 'Item', 'Year']
feature_cols = [
    col for col in df.columns
    if col not in excluded_columns
    and not col.startswith('Unnamed:')
]

if not item_dummy_cols:
    raise ValueError('No Item one-hot encoded columns were found.')

non_numeric_features = df[feature_cols].select_dtypes(exclude='number').columns.tolist()
if non_numeric_features:
    raise ValueError(f'Model features must be numeric: {non_numeric_features}')

target_col = 'log_yield'

print('Number of model features:', len(feature_cols))
print('Model features:')
print(feature_cols)
print('\nCrop categories:')
print([col.replace('Item_', '', 1) for col in item_dummy_cols])

display(df[feature_cols + [target_col]].head())

Number of model features: 15
Model features:
['rain_mm', 'pesticides_t', 'avg_temp', 'Area_frequency', 'Item_Cassava', 'Item_Maize', 'Item_Plantains and others', 'Item_Potatoes', 'Item_Rice, paddy', 'Item_Sorghum', 'Item_Soybeans', 'Item_Sweet potatoes', 'Item_Wheat', 'Item_Yams', 'years_since_1990']

Crop categories:
['Cassava', 'Maize', 'Plantains and others', 'Potatoes', 'Rice, paddy', 'Sorghum', 'Soybeans', 'Sweet potatoes', 'Wheat', 'Yams']


,rain_mm,pesticides_t,avg_temp,Area_frequency,Item_Cassava,Item_Maize,Item_Plantains and others,Item_Potatoes,"Item_Rice, paddy",Item_Sorghum,Item_Soybeans,Item_Sweet potatoes,Item_Wheat,Item_Yams,years_since_1990,log_yield
0,1485.0,121.0,16.37,0.003818,0,1,0,0,0,0,0,0,0,0,0,10.508159
1,1485.0,121.0,16.37,0.003818,0,0,0,1,0,0,0,0,0,0,0,11.107465
2,1485.0,121.0,16.37,0.003818,0,0,0,0,1,0,0,0,0,0,0,10.057624
3,1485.0,121.0,16.37,0.003818,0,0,0,0,0,1,0,0,0,0,0,9.433484
4,1485.0,121.0,16.37,0.003818,0,0,0,0,0,0,1,0,0,0,0,8.853665


## 4. Prepare the chronological training and testing sets

In [48]:
# Use the same chronological split as Task 2.2.
# Fit the scaler on training data only to prevent data leakage.
model_df = df[feature_cols + [target_col, 'Year']].copy()
model_df['Year'] = pd.to_numeric(model_df['Year'], errors='coerce')
model_df = model_df.dropna(subset=feature_cols + [target_col, 'Year'])

train_df = model_df[model_df['Year'] <= 2009].copy()
test_df = model_df[model_df['Year'].between(2010, 2013)].copy()

if train_df.empty or test_df.empty:
    raise ValueError('The chronological split produced an empty training or testing set.')

X_train = train_df[feature_cols].copy()
y_train = train_df[target_col].copy()
X_test = test_df[feature_cols].copy()
y_test = test_df[target_col].copy()

print('Training features:', X_train.shape)
print('Testing features:', X_test.shape)
print('Training target:', y_train.shape)
print('Testing target:', y_test.shape)
print('Training years:', train_df['Year'].min(), 'to', train_df['Year'].max())
print('Testing years:', test_df['Year'].min(), 'to', test_df['Year'].max())

Training features: (21305, 15)
Testing features: (4627, 15)
Training target: (21305,)
Testing target: (4627,)
Training years: 1990 to 2009
Testing years: 2010 to 2013


## 5. Train the complete regression pipeline

In [49]:
# Put scaling and Ridge regression into one pipeline.
# Ridge alpha=10 was the best-performing model in the Task 3.2 comparison.
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

final_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=10))
])

final_pipe.fit(X_train, y_train)

print('The regression pipeline has been trained.')
print(final_pipe)

The regression pipeline has been trained.
Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()), ('model', Ridge(alpha=10))])


## 6. Evaluate the pipeline on the test set

In [50]:
# Evaluate the model using the held-out years 2010–2013.
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

test_predictions = final_pipe.predict(X_test)

test_mae = mean_absolute_error(y_test, test_predictions)
test_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))
test_r2 = r2_score(y_test, test_predictions)

evaluation_results = pd.DataFrame([{
    'Model': 'Ridge Regression (alpha=10)',
    'Test_MAE': test_mae,
    'Test_RMSE': test_rmse,
    'Test_R2': test_r2
}])

display(evaluation_results)

,Model,Test_MAE,Test_RMSE,Test_R2
0,Ridge Regression (alpha=10),0.451414,0.597026,0.709904


## 7. Save the trained pipeline using Joblib

In [51]:
# Save the complete pipeline, including imputation, scaling, and regression.
import joblib

MODEL_PATH = MODEL_DIR / 'G03_model.joblib'
joblib.dump(final_pipe, MODEL_PATH)

print('Model saved to:', MODEL_PATH)
print('Model file size:', MODEL_PATH.stat().st_size, 'bytes')

Model saved to: /content/drive/MyDrive/CSC4110_G03 (1)/models/G03_model.joblib
Model file size: 2361 bytes


## 8. Reload the model and predict one test example

In [52]:
# Reload the saved model and verify that it can make a prediction.
loaded_pipe = joblib.load(MODEL_PATH)

one_test_row = X_test.iloc[[0]]
predicted_log_yield = float(loaded_pipe.predict(one_test_row)[0])

# Task 2.2 used the natural logarithm: log_yield = np.log(yield_hg_ha).
# Therefore, use the exponential function to return to hg/ha.
predicted_yield_hg_ha = float(np.exp(predicted_log_yield))
predicted_yield_tonnes_ha = predicted_yield_hg_ha / 10000

print('Test input:')
display(one_test_row)
print(f'Predicted log_yield: {predicted_log_yield:.6f}')
print(f'Predicted yield: {predicted_yield_hg_ha:,.2f} hg/ha')
print(f'Predicted yield: {predicted_yield_tonnes_ha:,.4f} tonnes/ha')

Test input:


,rain_mm,pesticides_t,avg_temp,Area_frequency,Item_Cassava,Item_Maize,Item_Plantains and others,Item_Potatoes,"Item_Rice, paddy",Item_Sorghum,Item_Soybeans,Item_Sweet potatoes,Item_Wheat,Item_Yams,years_since_1990
83,1485.0,1311.17,16.52,0.003818,0,1,0,0,0,0,0,0,0,0,20


Predicted log_yield: 10.307507
Predicted yield: 29,956.65 hg/ha
Predicted yield: 2.9957 tonnes/ha


## 9. Prepare the country and crop mappings for the application

In [53]:
# Recreate Area_frequency exactly as it was calculated in Task 2.2.
area_frequency = df_stage1['Area'].value_counts(normalize=True)

# Create a mapping from country name to its frequency in the original dataset.
area_frequency_mapping = {
    str(area): float(frequency)
    for area, frequency in area_frequency.items()
}

item_options = sorted([
    col.replace('Item_', '', 1) for col in item_dummy_cols
])

# Calculate default values for the environmental inputs.
environment_features = ['rain_mm', 'pesticides_t', 'avg_temp']
environment_defaults = {
    col: float(df[col].median())
    for col in environment_features
}

print('Number of countries:', len(area_frequency_mapping))
print('Crop options:', item_options)
print('Environmental defaults:', environment_defaults)
print('Example country mappings:', list(area_frequency_mapping.items())[:5])

Number of countries: 101
Crop options: ['Cassava', 'Maize', 'Plantains and others', 'Potatoes', 'Rice, paddy', 'Sorghum', 'Soybeans', 'Sweet potatoes', 'Wheat', 'Yams']
Environmental defaults: {'rain_mm': 1083.0, 'pesticides_t': 15373.0, 'avg_temp': 21.42}
Example country mappings: [('India', 0.13049514113836186), ('Brazil', 0.07149467838963443), ('Mexico', 0.04935986426037328), ('Australia', 0.037251272559000466), ('Pakistan', 0.037251272559000466)]


## 10. Save the application configuration

In [54]:
# Save the exact feature order and mappings needed by the Streamlit application.
app_config = {
    'feature_cols': feature_cols,
    'item_dummy_cols': item_dummy_cols,
    'item_options': item_options,
    'area_frequency_mapping': area_frequency_mapping,
    'environment_defaults': environment_defaults
}

CONFIG_PATH = MODEL_DIR / 'G03_app_config.json'
with open(CONFIG_PATH, 'w', encoding='utf-8') as config_file:
    json.dump(app_config, config_file, indent=2)

print('Application configuration saved to:', CONFIG_PATH)

Application configuration saved to: /content/drive/MyDrive/CSC4110_G03 (1)/models/G03_app_config.json


## 11. Create the Streamlit application

In [55]:
# Create the Streamlit app using the same features and mappings as model training.
app_source = r"""
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import streamlit as st

st.set_page_config(
    page_title="G03 Crop Yield Predictor",
    page_icon="🌾",
    layout="centered"
)

BASE_DIR = Path(__file__).resolve().parents[1]
MODEL_PATH = BASE_DIR / "models" / "G03_model.joblib"
CONFIG_PATH = BASE_DIR / "models" / "G03_app_config.json"

st.title("G03 Crop Yield Prediction")
st.subheader("CSC 4110 — Machine Learning Group Project")
st.write("**Group:** G03 — Crop Yield")
st.write(
    "**Group members:** Brayson Imbambi, Simon Njoroge, "
    "Jacob Chemning’wa, Austin Gatita, Agnes Watti"
)
st.write(
    "**Dataset:** FAO Crop Yield Dataset (`yield_df.csv`), "
    "cleaned as `clean_crops.csv`."
)

if not MODEL_PATH.exists() or not CONFIG_PATH.exists():
    st.error("The saved model or application configuration could not be found.")
    st.stop()

@st.cache_resource
def load_model():
    return joblib.load(MODEL_PATH)

@st.cache_data
def load_config():
    with open(CONFIG_PATH, "r", encoding="utf-8") as file:
        return json.load(file)

model = load_model()
config = load_config()

feature_cols = config["feature_cols"]
item_dummy_cols = config["item_dummy_cols"]
item_options = config["item_options"]
area_frequency_mapping = config["area_frequency_mapping"]
environment_defaults = config["environment_defaults"]

st.markdown("### Enter crop and location details")

selected_item = st.selectbox("Item (crop)", item_options)
area_options = sorted(area_frequency_mapping.keys())
selected_area = st.selectbox("Area (country)", area_options)

year = st.number_input(
    "Year",
    min_value=1990,
    max_value=2035,
    value=2023,
    step=1
)

st.markdown("### Environmental information")

rain_mm = st.number_input(
    "Annual rainfall (mm)",
    min_value=0.0,
    value=float(environment_defaults["rain_mm"]),
    step=10.0
)

pesticides_t = st.number_input(
    "Pesticides (tonnes)",
    min_value=0.0,
    value=float(environment_defaults["pesticides_t"]),
    step=10.0
)

avg_temp = st.number_input(
    "Average temperature (°C)",
    value=float(environment_defaults["avg_temp"]),
    step=0.1
)

if st.button("Predict crop yield", type="primary"):
    # Create one row with the same feature names and order used during training.
    row = {feature: 0.0 for feature in feature_cols}

    # Set the crop one-hot encoded feature.
    selected_dummy = "Item_" + selected_item
    if selected_dummy in row:
        row[selected_dummy] = 1.0

    # Set the country frequency calculated from the original Task 2.2 data.
    row["Area_frequency"] = float(area_frequency_mapping[selected_area])

    # Set the year-based and environmental features.
    if "years_since_1990" in row:
        row["years_since_1990"] = float(year - 1990)

    if "rain_mm" in row:
        row["rain_mm"] = float(rain_mm)

    if "pesticides_t" in row:
        row["pesticides_t"] = float(pesticides_t)

    if "avg_temp" in row:
        row["avg_temp"] = float(avg_temp)

    input_df = pd.DataFrame([row], columns=feature_cols)

    try:
        predicted_log_yield = float(model.predict(input_df)[0])

        # The group created log_yield using the natural logarithm.
        predicted_hg_ha = float(np.exp(predicted_log_yield))
        predicted_tonnes_ha = predicted_hg_ha / 10000.0

        st.success(f"Prediction completed for {selected_item} in {selected_area}.")
        st.metric("Estimated crop yield", f"{predicted_hg_ha:,.2f} hg/ha")
        st.metric("Estimated crop yield", f"{predicted_tonnes_ha:,.4f} tonnes/ha")
        st.caption(
            "This is a statistical estimate based on the group's trained model, "
            "not a guaranteed agricultural outcome."
        )
    except Exception as error:
        st.error(f"Prediction failed: {error}")
"""

APP_PATH = APP_DIR / 'app.py'
APP_PATH.write_text(app_source, encoding='utf-8')

print('Streamlit application created:', APP_PATH)

Streamlit application created: /content/drive/MyDrive/CSC4110_G03 (1)/app/app.py


## 12. Create the application requirements file

In [56]:
# List the packages needed to run the Streamlit application.
requirements = '''streamlit
scikit-learn
joblib
pandas
numpy
'''

REQUIREMENTS_PATH = APP_DIR / 'requirements.txt'
REQUIREMENTS_PATH.write_text(requirements, encoding='utf-8')

print('Requirements file created:', REQUIREMENTS_PATH)
print(REQUIREMENTS_PATH.read_text())

Requirements file created: /content/drive/MyDrive/CSC4110_G03 (1)/app/requirements.txt
streamlit
scikit-learn
joblib
pandas
numpy



## 13. Run the Streamlit application and capture the screenshot

The notebook creates the application files. To run the application on your computer, open a terminal in the project folder and execute:

```bash
pip install -r app/requirements.txt
streamlit run app/app.py
```

When the application opens:
1. Select a crop.
2. Select a country.
3. Enter the year, rainfall, pesticide use and average temperature.
4. Click **Predict crop yield**.
5. Capture a screenshot showing the running app and its prediction.
6. Save the screenshot as `outputs/app_screenshot.png`.

The app uses the country-to-frequency mapping calculated from `stage1_crops.csv` using the same `value_counts(normalize=True)` method as Task 2.2.

## 14. Verify the required deliverables

In [57]:
# Check that the model and application files have been created.
required_files = [
    MODEL_PATH,
    CONFIG_PATH,
    APP_DIR / 'app.py',
    APP_DIR / 'requirements.txt'
]

for file_path in required_files:
    print(('FOUND: ' if file_path.exists() else 'MISSING: ') + str(file_path))

screenshot_path = OUTPUT_DIR / 'app_screenshot.png'
print('\nScreenshot location:', screenshot_path)
print('Capture the screenshot after running the Streamlit application.')

FOUND: /content/drive/MyDrive/CSC4110_G03 (1)/models/G03_model.joblib
FOUND: /content/drive/MyDrive/CSC4110_G03 (1)/models/G03_app_config.json
FOUND: /content/drive/MyDrive/CSC4110_G03 (1)/app/app.py
FOUND: /content/drive/MyDrive/CSC4110_G03 (1)/app/requirements.txt

Screenshot location: /content/drive/MyDrive/CSC4110_G03 (1)/outputs/app_screenshot.png
Capture the screenshot after running the Streamlit application.


In [58]:
!pip install -r /content/drive/MyDrive/CSC4110_G03 (1)/app/requirements.txt

/bin/bash: -c: line 1: syntax error near unexpected token `('
/bin/bash: -c: line 1: `pip install -r /content/drive/MyDrive/CSC4110_G03 (1)/app/requirements.txt'


In [59]:
!streamlit run /content/drive/MyDrive/CSC4110_G03/app/app.py --server.port 8501 --server.address 0.0.0.0 > /content/streamlit.log 2>&1 &

In [60]:
!python -m pip install streamlit

In [61]:
!python -m streamlit --version

Streamlit, version 1.67.0


In [62]:
import subprocess
import time

log = open("/content/streamlit_new.log", "w")

app = subprocess.Popen(
    [
        "python", "-m", "streamlit", "run",
        "/content/drive/MyDrive/CSC4110_G03 (1)/app/app.py",
        "--server.port", "8502",
        "--server.address", "0.0.0.0"
    ],
    stdout=log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

time.sleep(10)

print("Streamlit process running:", app.poll() is None)

Streamlit process running: False


In [63]:
from google.colab import output

url = output.eval_js("google.colab.kernel.proxyPort(8502)")

print("Your application link:", url)

Your application link: https://8502-m-s-usw3b7-1xo63jjmjodnp-b.us-west3.prod.r.colab.dev


In [64]:
import requests
import subprocess

print("1. Checking Streamlit response...")

try:
    response = requests.get("http://127.0.0.1:8502", timeout=10)
    print("HTTP status:", response.status_code)
except Exception as e:
    print("Connection error:", e)

print("\n2. Checking Streamlit process...")
!ps aux | grep '[s]treamlit'

print("\n3. Checking Streamlit logs...")
!tail -40 /content/streamlit_new.log

1. Checking Streamlit response...
HTTP status: 200

2. Checking Streamlit process...
root       16138  0.7  1.7 1626948 234572 ?      Ssl  19:17   0:35 python3 -m streamlit run /content/drive/MyDrive/CSC4110_G03 (1)/app/app.py --server.port 8502 --server.address 0.0.0.0 --server.enableCORS false --server.enableXsrfProtection false

3. Checking Streamlit logs...


2026-11-26 20:37:26.913 Port 8502 is not available


In [65]:
import subprocess
import time
import os
import psutil

# Stop the existing Streamlit process on port 8502
for process in psutil.process_iter(['pid', 'cmdline']):
    try:
        command = ' '.join(process.info['cmdline'] or [])
        if 'streamlit' in command and '8502' in command:
            if process.info['pid'] != os.getpid():
                process.terminate()
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        pass

time.sleep(3)

# Start Streamlit with Colab-compatible security settings
log = open("/content/streamlit_new.log", "w")

app = subprocess.Popen(
    [
        "python", "-m", "streamlit", "run",
        "/content/drive/MyDrive/CSC4110_G03 (1)/app/app.py",
        "--server.port", "8502",
        "--server.address", "0.0.0.0",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false"
    ],
    stdout=log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

time.sleep(10)

print("Streamlit running:", app.poll() is None)

Streamlit running: True


In [66]:
from google.colab import output

url = output.eval_js("google.colab.kernel.proxyPort(8502)")

print("Application link:", url)

Application link: https://8502-m-s-usw3b7-1xo63jjmjodnp-b.us-west3.prod.r.colab.dev


## 15. Written answer

The group workflow began by collecting the FAO crop-yield dataset and preparing it through data cleaning, logarithmic target transformation, country-frequency encoding and one-hot encoding of crop types. Exploratory data analysis was then carried out to examine crop distributions, feature relationships and the distribution of crop yield. The data was split chronologically into training records from 1990–2009 and testing records from 2010–2013, with the scaler fitted only on the training data. Regression and classification models were trained and evaluated, after which the selected Ridge regression model was retrained as a complete pipeline and saved using Joblib. Finally, the saved model was integrated into a Streamlit application, tested with user inputs and prepared for deployment.

The complete pipeline was saved instead of only the Ridge estimator because the model requires the same preprocessing used during training. Keeping imputation, scaling and prediction together ensures that new application inputs are transformed consistently before the model generates a prediction.

## 16. Submission checklist

- [ ] Save this notebook as `notebooks/G03_U9.1_packaging_app.ipynb`.
- [ ] Run all cells from top to bottom.
- [ ] Confirm `final_pipe` is created.
- [ ] Confirm `models/G03_model.joblib` is saved and can be reloaded.
- [ ] Confirm `models/G03_app_config.json` is created.
- [ ] Confirm `app/app.py` and `app/requirements.txt` are created.
- [ ] Run the Streamlit application and save the real screenshot as `outputs/app_screenshot.png`.
- [ ] Export the notebook to PDF if required and submit the notebook, PDF and Colab link to Moodle.